# Demostración: delegación del orquestador multi-agente

Este notebook ejecuta el grafo con dos solicitudes:

1. **Camino principal:** el supervisor envía la tarea al investigador, después al analista, y la cierra cuando el validador aprueba.
2. **Refinamiento:** el supervisor intenta cerrar con un análisis desactualizado, el validador lo rechaza y el supervisor manda a rehacer el análisis.

> **Modo de ejecución.** Con `DEMO_MODO=guionizado`, que es el valor por defecto, las decisiones del supervisor y los textos de los agentes están **escritos a mano** en `tests/demo_scripts.py`. El grafo, las herramientas, la base SQLite, las estadísticas y el validador sí se ejecutan de verdad. Con `DEMO_MODO=real` y una clave en `.env`, decide un LLM real. Las salidas guardadas en este notebook son del modo guionizado.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path[:0] = [str(ROOT), str(ROOT / 'tests')]

from dotenv import load_dotenv

load_dotenv(ROOT / '.env')
MODO = os.getenv('DEMO_MODO', 'guionizado')

from demo_scripts import coffee_with_refinement, headphones_comparison
from graph import build_graph, run_request
from llm import build_models
from main import DEMO_REQUESTS
from trace_log import format_delegation

def models_for(scripted):
    return build_models() if MODO == 'real' else scripted()

print('Modo:', MODO)

Modo: guionizado


## El grafo

Diagrama Mermaid generado por LangGraph a partir del grafo compilado.

In [2]:
print(build_graph(headphones_comparison()).get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	researcher(researcher)
	analyst(analyst)
	validator(validator)
	synthesizer(synthesizer)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analyst --> supervisor;
	researcher --> supervisor;
	supervisor -.-> analyst;
	supervisor -.-> researcher;
	supervisor -.-> validator;
	validator -.-> supervisor;
	validator -.-> synthesizer;
	synthesizer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Escenario 1: investigador, luego analista, luego cierre

In [3]:
comparison_models = models_for(headphones_comparison)
state = await run_request(build_graph(comparison_models), DEMO_REQUESTS[0])
print('Solicitud:', DEMO_REQUESTS[0])
print('\nDelegación:')
print(format_delegation(state))
print('\nRespuesta final:')
print(state['final_answer'])

Solicitud: Compara lo que opinan los clientes de los audífonos AuraSound X2 y Nimbus Pro, y dime cuál recomiendas.

Delegación:
 1. [supervisor] -> researcher: Faltan los datos de ambos productos.
 2. [researcher] Investigación v1: AuraSound X2 (6 reseñas), Nimbus Pro (6 reseñas)
 3. [supervisor] -> analyst: La investigación ya cubre los dos productos.
 4. [analyst] Análisis v1 sobre investigación v1: AuraSound X2 (promedio 4.17), Nimbus Pro (promedio 2.5)
 5. [supervisor] -> finish: La investigación y el análisis vigentes cubren ambos productos.
 6. [validator] Aprobado
 7. [synthesizer] Respuesta final redactada

Respuesta final:
Recomiendo los AuraSound X2. Tienen un promedio de 4,17 estrellas en 6 reseñas, el 83,33% son de 4 o 5 estrellas y 5 de 6 reseñas tienen tono positivo. Los Nimbus Pro promedian 2,5 en 6 reseñas, solo el 16,67% llega a 4 o 5 estrellas y su sentimiento es mayormente neutro o negativo (1 positiva, 3 neutras y 2 negativas). Estas conclusiones salen de las reseña

### Lo que contiene el estado compartido

Cada agente escribe solo en su parte del estado: el investigador en `research`, el analista en `analysis` y el validador en `validation`. Las cifras salen de las herramientas, no del texto del LLM.

In [4]:
for product in state['research'].productos:
    print(f'{product.nombre}: {len(product.resenas)} reseñas (investigación v{state["research"].version})')
for item in state['analysis'].productos:
    stats, sentiment = item.estadisticas, item.sentimiento
    print(f'{item.nombre}: promedio {stats.promedio}, {stats.porcentaje_positivas}% con 4-5 estrellas, '
          f'sentimiento +{sentiment.positivas} / ={sentiment.neutras} / -{sentiment.negativas}')
print('Validación aprobada:', state['validation'].aprobado)

AuraSound X2: 6 reseñas (investigación v1)
Nimbus Pro: 6 reseñas (investigación v1)
AuraSound X2: promedio 4.17, 83.33% con 4-5 estrellas, sentimiento +5 / =0 / -1
Nimbus Pro: promedio 2.5, 16.67% con 4-5 estrellas, sentimiento +1 / =3 / -2
Validación aprobada: True


### Contexto que recibió cada especialista

Para evitar la contaminación de contexto, los especialistas no reciben el historial completo: solo la instrucción del supervisor y, en el caso del analista, la lista de productos investigados. Esta celda solo funciona en modo guionizado, porque usa el registro del modelo de prueba.

In [5]:
if MODO != 'real':
    for role in ('researcher', 'analyst'):
        first_call = getattr(comparison_models, role).received[0]
        print(f'--- {role}: {len(first_call)} mensajes')
        print(first_call[-1].content)

--- researcher: 2 mensajes
Instrucción del supervisor: Busca los audífonos AuraSound X2 y Nimbus Pro y obtén todas sus reseñas.
--- analyst: 2 mensajes
Instrucción del supervisor: Calcula estadísticas y sentimiento de los dos productos investigados.

Productos investigados (investigación v1):
- producto_id 1: AuraSound X2 (6 reseñas)
- producto_id 2: Nimbus Pro (6 reseñas)


## Escenario 2: el validador detiene un cierre prematuro

El guion simula un supervisor que pide finalizar después de ampliar la investigación, pero sin rehacer el análisis. El validador detecta que el análisis usa una versión vieja de la investigación.

In [6]:
state = await run_request(build_graph(models_for(coffee_with_refinement)), DEMO_REQUESTS[1])
print('Solicitud:', DEMO_REQUESTS[1])
print('\nDelegación:')
print(format_delegation(state))
print('\nRespuesta final:')
print(state['final_answer'])

Solicitud: ¿Cómo les va en las reseñas a la cafetera Brava 300 y al molino Molienda Fina?

Delegación:
 1. [supervisor] -> researcher: Todavía no hay investigación.
 2. [researcher] Investigación v1: Brava 300 (5 reseñas)
 3. [supervisor] -> analyst: Ya hay datos de la Brava 300.
 4. [analyst] Análisis v1 sobre investigación v1: Brava 300 (promedio 4.4)
 5. [supervisor] -> researcher: La solicitud también pide el molino Molienda Fina.
 6. [researcher] Investigación v2: Brava 300 (5 reseñas), Molienda Fina (4 reseñas)
 7. [supervisor] -> finish: Ya están los dos productos.
 8. [validator] Rechazado: El análisis usa la investigación v1, pero la vigente es la v2. Faltan las estadísticas de Molienda Fina. Falta el sentimiento de Molienda Fina.
 9. [supervisor] -> analyst: La validación detectó un análisis desactualizado.
10. [analyst] Análisis v2 sobre investigación v2: Brava 300 (promedio 4.4), Molienda Fina (promedio 3.25)
11. [supervisor] -> finish: El análisis ya usa la investigación v